# 17_threshold_sweep_val.ipynb
## Giai Đoạn 5 (Tuần 9): Tối Ưu Hóa Ngưỡng Tin Cậy (Confidence Threshold Sweep) Trên Tập Validation

### 1. Pre-flight & Thiết Lập Ràng Buộc Đánh Giá Giai Đoạn 5 (Tuần 9)
- **B0 checkpoint dùng**: `runs/B0_100ep/weights/best.pt` (Baseline YOLOv8n chuẩn 100 epochs, mAP@0.5: 0.7471).
- **B1 checkpoint dùng**: `runs/B1_100ep/weights/best.pt` (Mô hình YOLOv8n-P2 100 epochs, mAP@0.5: 0.7034).
- **Nguyên tắc chọn threshold**: Mọi quá trình tối ưu hóa ngưỡng tin cậy (Confidence Threshold Tuning / F1-Max / F2-Max / Cost-Sensitive Thresholding) **CHỈ ĐƯỢC PHÉP THỰC HIỆN TRÊN TẬP VALIDATION (`data/splits/val.txt`)**.
- **Nguyên tắc bảo toàn tập Test**: Tập kiểm thử độc lập (`data/splits/test.txt` - 480 ảnh) **CHỈ ĐƯỢC PHÉP MỞ KHÓA Ở TUẦN 10** sau khi toàn bộ quy tắc chọn ngưỡng của các mô hình đã được cố định hoàn toàn.

### 2. Mục tiêu nghiên cứu
1. Quét lưới confidence threshold từ `0.01` đến `0.95` (bước nhảy `0.01`).
2. Tối ưu điểm làm việc cân bằng theo $F_1$-Score toàn cục và theo từng lớp khuyết tật.
3. Tối ưu ngưỡng nhạy cảm chi phí theo $F_2$-Score ($eta = 2$, ưu tiên Recall) cho các lớp lỗi chí mạng (`open_circuit`, `short`) để giảm thiểu tỷ lệ bỏ sót lỗi (False Negative Rate).


### 2. Import các thư viện cần thiết
Nạp đầy đủ các module theo tiêu chuẩn: `ultralytics`, `pandas`, `numpy`, `pathlib`, `yaml`, `matplotlib`, `json`, `datetime`, `PIL.Image`, `collections.defaultdict`.


In [1]:
from collections import defaultdict
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import yaml

def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
os.chdir(ROOT)

print("=== THIẾT LẬP MÔI TRƯỜNG ===")
print(f"Project root     : {ROOT}")
print(f"Working directory: {os.getcwd()}")
print(f"Timestamp (UTC)  : {datetime.now(timezone.utc).isoformat()}")


=== THIẾT LẬP MÔI TRƯỜNG ===
Project root     : /Users/Project/kltn-pcb
Working directory: /Users/Project/kltn-pcb
Timestamp (UTC)  : 2026-09-25T06:05:00.469782+00:00


### 3. Đọc file cấu hình và tập dữ liệu Validation
- Đọc `configs/base.yaml`
- Đọc `data/data.yaml`
- Đọc danh sách ảnh từ `data/splits/val.txt` (368 ảnh)
- Kiểm tra tính độc lập tuyệt đối với tập test (không nạp test set vào quá trình quét ngưỡng)


In [2]:
CONFIG_BASE_PATH = ROOT / "configs" / "base.yaml"
DATA_YAML_PATH   = ROOT / "data" / "data.yaml"
VAL_TXT_PATH     = ROOT / "data" / "splits" / "val.txt"
TEST_TXT_PATH    = ROOT / "data" / "splits" / "test.txt"

# 1. Đọc configs/base.yaml
with open(CONFIG_BASE_PATH, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

# 2. Đọc data/data.yaml
with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

# 3. Đọc danh sách ảnh validation
with open(VAL_TXT_PATH, "r", encoding="utf-8") as f:
    val_img_paths = [line.strip() for line in f if line.strip()]

# 4. Kiểm tra an toàn: Đảm bảo tập test hoàn toàn tách biệt, không bị dùng
assert TEST_TXT_PATH.is_file(), f"LỖI: Không tìm thấy {TEST_TXT_PATH}"
with open(TEST_TXT_PATH, "r", encoding="utf-8") as f:
    test_img_paths = [line.strip() for line in f if line.strip()]

overlap = set(val_img_paths) & set(test_img_paths)
assert len(overlap) == 0, f"RÒ RỈ DỮ LIỆU: Có {len(overlap)} ảnh trùng lặp giữa val và test!"
del test_img_paths  # Xóa khỏi bộ nhớ ngay lập tức để đảm bảo cô lập

print("=== ĐÃ TẢI CẤU HÌNH VÀ TẬP DỮ LIỆU ===")
print(f"Cấu hình gốc base.yaml: {CONFIG_BASE_PATH.relative_to(ROOT)}")
print(f"Cấu hình dữ liệu      : {DATA_YAML_PATH.relative_to(ROOT)}")
print(f"Tập kiểm định (val)   : {VAL_TXT_PATH.relative_to(ROOT)} ({len(val_img_paths)} ảnh)")
print(f"Danh mục 6 lớp lỗi    : {data_cfg.get('names', {})}")
print(f"Kiểm tra cách ly Test : [PASS] 0 ảnh test bị rò rỉ vào validation.")


=== ĐÃ TẢI CẤU HÌNH VÀ TẬP DỮ LIỆU ===
Cấu hình gốc base.yaml: configs/base.yaml
Cấu hình dữ liệu      : data/data.yaml
Tập kiểm định (val)   : data/splits/val.txt (368 ảnh)
Danh mục 6 lớp lỗi    : {0: 'missing_hole', 1: 'mouse_bite', 2: 'open_circuit', 3: 'short', 4: 'spur', 5: 'spurious_copper'}
Kiểm tra cách ly Test : [PASS] 0 ảnh test bị rò rỉ vào validation.


### 4. Khai báo mô hình và cấu hình tham số Threshold Sweep
- **Mô hình B0**: Checkpoint `runs/B0_100ep/weights/best.pt`
- **Mô hình B1**: Checkpoint `runs/B1_100ep/weights/best.pt`
- **Lớp lỗi chí mạng**: `critical_classes = ["open_circuit", "short"]`
- **Hệ số F-beta**: `beta = 2` (tập trung phạt False Negative đối với lỗi mạch nghiêm trọng)
- **IoU threshold**: `iou_threshold = 0.5`
- **Dải ngưỡng tin cậy**: `conf_values` từ `0.01` đến `0.95` (bước `0.01`, tổng cộng 95 mốc giá trị)


In [3]:
# ── Khai báo đường dẫn Checkpoint của 2 mô hình ─────────────────────────────
MODEL_B0_PATH = ROOT / "runs" / "B0_100ep" / "weights" / "best.pt"
MODEL_B1_PATH = ROOT / "runs" / "B1_100ep" / "weights" / "best.pt"

# Kiểm tra sự tồn tại của cả hai checkpoint
assert MODEL_B0_PATH.is_file(), f"❌ THIẾU CHECKPOINT B0: {MODEL_B0_PATH}"
assert MODEL_B1_PATH.is_file(), f"❌ THIẾU CHECKPOINT B1: {MODEL_B1_PATH}"

models_dict = {
    "B0 (YOLOv8n-Standard)": MODEL_B0_PATH,
    "B1 (YOLOv8n-P2)"      : MODEL_B1_PATH,
}

# ── Khai báo siêu tham số Threshold Sweep ───────────────────────────────────
critical_classes = ["open_circuit", "short"]
beta = 2
iou_threshold = 0.5
conf_values = np.round(np.arange(0.01, 0.951, 0.01), 2)

# ── In báo cáo xác nhận Setup Hoàn Tất ───────────────────────────────────────
print("=" * 80)
print("XÁC NHẬN SETUP HOÀN TẤT CHO THRESHOLD SWEEP (GIAI ĐOẠN 5 - TUẦN 9)")
print("=" * 80)
print("1. Danh sách mô hình đánh giá:")
for name, path in models_dict.items():
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"   - {name}: {path.relative_to(ROOT)} ({size_mb:.2f} MB)")

print("\n2. Tập dữ liệu đánh giá ngưỡng:")
print(f"   - Split                 : Validation only ({VAL_TXT_PATH.relative_to(ROOT)})")
print(f"   - Số lượng ảnh val      : {len(val_img_paths)} ảnh")
print(f"   - Trạng thái tập test   : Hoàn toàn CÔ LẬP (chỉ dùng ở Tuần 10)")

print("\n3. Cấu hình quét ngưỡng & chỉ số mục tiêu:")
print(f"   - Critical classes      : {critical_classes}")
print(f"   - Hệ số F-beta          : beta = {beta} (F2-Score tối ưu độ nhạy / Recall)")
print(f"   - IoU threshold         : {iou_threshold}")
print(f"   - Dải ngưỡng tin cậy    : {len(conf_values)} giá trị từ {conf_values[0]:.2f} đến {conf_values[-1]:.2f} (step = 0.01)")
print("=" * 80)


XÁC NHẬN SETUP HOÀN TẤT CHO THRESHOLD SWEEP (GIAI ĐOẠN 5 - TUẦN 9)
1. Danh sách mô hình đánh giá:
   - B0 (YOLOv8n-Standard): runs/B0_100ep/weights/best.pt (5.95 MB)
   - B1 (YOLOv8n-P2): runs/B1_100ep/weights/best.pt (5.93 MB)

2. Tập dữ liệu đánh giá ngưỡng:
   - Split                 : Validation only (data/splits/val.txt)
   - Số lượng ảnh val      : 368 ảnh
   - Trạng thái tập test   : Hoàn toàn CÔ LẬP (chỉ dùng ở Tuần 10)

3. Cấu hình quét ngưỡng & chỉ số mục tiêu:
   - Critical classes      : ['open_circuit', 'short']
   - Hệ số F-beta          : beta = 2 (F2-Score tối ưu độ nhạy / Recall)
   - IoU threshold         : 0.5
   - Dải ngưỡng tin cậy    : 95 giá trị từ 0.01 đến 0.95 (step = 0.01)


### 5. Sinh / Tái Sử Dụng (Cache) Dự Đoán Chi Tiết Trên Validation Set
- Với mỗi mô hình ($B_0$ và $B_1$), tiến hành inference trên toàn bộ 368 ảnh trong `val.txt` với ngưỡng tin cậy sàn `conf = 0.01` và `iou = 0.7`.
- **Mục đích lưu Cache**: Trích xuất toàn bộ ứng viên bounding box tiềm năng một lần duy nhất, cho phép quét ngưỡng offline qua 95 mốc `conf_values` mà không cần gọi lại `model.predict()` 95 lần.
- **Cấu trúc trường thông tin xuất ra**:
  - `model_code`: Mã mô hình (`B0` hoặc `B1`)
  - `image_path`: Đường dẫn tương đối của ảnh từ project root
  - `image_name`: Tên file ảnh (e.g. `01_missing_hole_01.jpg`)
  - `class_id`: ID lớp khuyết tật (0 đến 5)
  - `class_name`: Tên lớp khuyết tật tương ứng
  - `confidence`: Điểm tin cậy dự đoán (float)
  - `xmin`, `ymin`, `xmax`, `ymax`: Tọa độ pixel của bounding box
- **Đường dẫn lưu trữ cache**:
  - `results/B0/val_predictions_for_threshold.csv`
  - `results/B1/val_predictions_for_threshold.csv`
- **Cơ chế tái sử dụng**: Nếu file cache đã tồn tại, đọc lại file và in thông báo `[REUSE CACHE]` để bảo toàn tài nguyên.


In [4]:
val_full_paths = [str(ROOT / p) for p in val_img_paths]
raw_names = data_cfg.get("names", {})
names_dict = {int(k): v for k, v in raw_names.items()}

models_to_predict = [
    ("B0", MODEL_B0_PATH, ROOT / "results" / "B0" / "val_predictions_for_threshold.csv"),
    ("B1", MODEL_B1_PATH, ROOT / "results" / "B1" / "val_predictions_for_threshold.csv"),
]

val_predictions_dict = {}

for model_code, model_path, cache_csv in models_to_predict:
    cache_csv.parent.mkdir(parents=True, exist_ok=True)
    
    if cache_csv.is_file():
        print(f"ℹ️  [REUSE CACHE] Mô hình {model_code}: Tìm thấy file cache tại {cache_csv.relative_to(ROOT)}")
        df_pred = pd.read_csv(cache_csv)
        print(f"   Đã đọc thành công {len(df_pred):,} bounding boxes từ cache.")
    else:
        print(f"🚀 [INFERENCE] Mô hình {model_code}: Đang chạy inference trên {len(val_full_paths)} ảnh validation (conf=0.01, iou=0.7)...")
        model = YOLO(str(model_path))
        preds_batch = model.predict(
            source=val_full_paths,
            conf=0.01,
            iou=0.7,
            imgsz=int(base_cfg.get("imgsz", 640)),
            save=False,
            verbose=False,
        )
        
        pred_records = []
        for img_rel, res in zip(val_img_paths, preds_batch):
            img_name = Path(img_rel).name
            boxes = res.boxes
            if boxes is not None and len(boxes) > 0:
                for b in boxes:
                    cls_id = int(b.cls[0].item())
                    conf_score = float(b.conf[0].item())
                    xyxy = b.xyxy[0].tolist()
                    pred_records.append({
                        "model_code": model_code,
                        "image_path": str(img_rel),
                        "image_name": img_name,
                        "class_id": cls_id,
                        "class_name": names_dict.get(cls_id, f"class_{cls_id}"),
                        "confidence": round(conf_score, 6),
                        "xmin": round(xyxy[0], 2),
                        "ymin": round(xyxy[1], 2),
                        "xmax": round(xyxy[2], 2),
                        "ymax": round(xyxy[3], 2),
                    })
        
        df_pred = pd.DataFrame(pred_records)
        df_pred.to_csv(cache_csv, index=False)
        print(f"✅ [SAVED] Mô hình {model_code}: Đã lưu {len(df_pred):,} predictions vào {cache_csv.relative_to(ROOT)}")
    
    val_predictions_dict[model_code] = df_pred

print("\n" + "=" * 80)
print("TỔNG KẾT PREDICTIONS VALIDATION ĐÃ CACHE:")
for m_code, df_p in val_predictions_dict.items():
    print(f"  - Model {m_code}: {len(df_p):,} boxes | Conf min: {df_p['confidence'].min():.4f} | Conf max: {df_p['confidence'].max():.4f}")
print("=" * 80)


ℹ️  [REUSE CACHE] Mô hình B0: Tìm thấy file cache tại results/B0/val_predictions_for_threshold.csv
   Đã đọc thành công 4,530 bounding boxes từ cache.
ℹ️  [REUSE CACHE] Mô hình B1: Tìm thấy file cache tại results/B1/val_predictions_for_threshold.csv
   Đã đọc thành công 7,845 bounding boxes từ cache.

TỔNG KẾT PREDICTIONS VALIDATION ĐÃ CACHE:
  - Model B0: 4,530 boxes | Conf min: 0.0100 | Conf max: 0.8600
  - Model B1: 7,845 boxes | Conf min: 0.0100 | Conf max: 0.9159


### 6. Thống Kê Phân Bố Bounding Box Dự Đoán Theo Lớp Khuyết Tật (Sàn Conf = 0.01)


In [5]:
summary_rows = []
for model_code, df_p in val_predictions_dict.items():
    counts = df_p["class_name"].value_counts().to_dict()
    row = {"model_code": model_code, "total_boxes": len(df_p)}
    for cid in range(6):
        cname = names_dict[cid]
        row[cname] = counts.get(cname, 0)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("=== PHÂN BỐ DỰ ĐOÁN THEO LỚP (CONF >= 0.01) ===")
print(df_summary.to_string(index=False))


=== PHÂN BỐ DỰ ĐOÁN THEO LỚP (CONF >= 0.01) ===
model_code  total_boxes  missing_hole  mouse_bite  open_circuit  short  spur  spurious_copper
        B0         4530          1167         404           585    899  1113              362
        B1         7845          4803         315           536    747   847              597


### 7. Tải Ground Truth Tập Validation từ Nhãn YOLO (Phase 4)
- **Quy trình trích xuất**:
  1. Đọc danh sách ảnh từ `data/splits/val.txt` (368 ảnh).
  2. Xác định đường dẫn file nhãn tương ứng (`.txt`) trong `data/raw/PKU-Market-PCB(Data enhanced version)/*/labels/` (hoặc `data/processed/labels/*/`).
  3. Đọc kích thước gốc của ảnh `(W, H)` bằng `PIL.Image`.
  4. Chuyển đổi nhãn chuẩn hóa YOLO `[class_id, x_center, y_center, width, height]` sang tọa độ pixel tuyệt đối:
     $$x_{min} = (x_c - w/2) 	imes W, \quad y_{min} = (y_c - h/2) 	imes H$$
     $$x_{max} = (x_c + w/2) 	imes W, \quad y_{max} = (y_c + h/2) 	imes H$$
  5. Tính toán diện tích theo tỷ lệ không gian đầu vào mô hình $640 	imes 640$:
     $$scale = \min\left(rac{640}{W}, rac{640}{H}ight)$$
     $$w_{px} = (x_{max} - x_{min}) 	imes scale, \quad h_{px} = (y_{max} - y_{min}) 	imes scale, \quad area_{px} = w_{px} 	imes h_{px}$$
  6. Phân bổ size bin theo quy tắc COCO đã khóa cố định:
     - `small`: $area_{px} < 32^2 = 1024$
     - `medium`: $32^2 \le area_{px} \le 96^2 = 9216$
     - `large`: $area_{px} > 96^2 = 9216$
- **DataFrame Ground Truth** chứa 10 trường: `image_path`, `image_name`, `class_id`, `class_name`, `xmin`, `ymin`, `xmax`, `ymax`, `area_px`, `size_bin`.


In [6]:
gt_records = []

for img_rel in val_img_paths:
    img_path = ROOT / img_rel
    img_name = img_path.name
    
    # Tìm file nhãn tương ứng (ưu tiên raw sau đó fallback processed)
    lbl_path = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
    if not lbl_path.is_file():
        lbl_path = ROOT / "data/processed/labels" / img_path.parent.parent.name / (img_path.stem + ".txt")
    
    assert lbl_path.is_file(), f"❌ Không tìm thấy label file cho {img_path}"
    
    # Đọc kích thước ảnh gốc
    with Image.open(img_path) as im:
        W, H = im.size
    
    # Scale cố định theo imgsz=640 của quy chuẩn khóa Tuần 3
    scale = min(640.0 / W, 640.0 / H)
    
    with open(lbl_path, "r", encoding="utf-8") as fp:
        lines = [line.strip() for line in fp if line.strip()]
        
    for line in lines:
        parts = line.split()
        cls_id = int(parts[0])
        xc, yc, w_norm, h_norm = map(float, parts[1:5])
        
        # Tọa độ pixel tuyệt đối
        xmin = (xc - w_norm / 2.0) * W
        ymin = (yc - h_norm / 2.0) * H
        xmax = (xc + w_norm / 2.0) * W
        ymax = (yc + h_norm / 2.0) * H
        
        # Kích thước box quy về không gian mô hình 640x640
        w_px = (xmax - xmin) * scale
        h_px = (ymax - ymin) * scale
        area_px = w_px * h_px
        
        # Phân loại kích thước theo rule đã khóa
        if area_px < 32**2:
            size_bin = "small"
        elif area_px <= 96**2:
            size_bin = "medium"
        else:
            size_bin = "large"
            
        gt_records.append({
            "image_path": str(img_rel),
            "image_name": img_name,
            "class_id": cls_id,
            "class_name": names_dict.get(cls_id, f"class_{cls_id}"),
            "xmin": round(xmin, 2),
            "ymin": round(ymin, 2),
            "xmax": round(xmax, 2),
            "ymax": round(ymax, 2),
            "area_px": round(area_px, 2),
            "size_bin": size_bin,
        })

df_val_gt = pd.DataFrame(gt_records)

# Lưu cache bảng ground truth để tái lập thuận tiện
VAL_GT_CSV = ROOT / "results" / "val_ground_truth.csv"
df_val_gt.to_csv(VAL_GT_CSV, index=False)

print("=" * 80)
print(f"XÁC NHẬN TẢI THÀNH CÔNG GROUND TRUTH VALIDATION: {len(df_val_gt):,} BOXES")
print(f"Đã lưu cache tại: {VAL_GT_CSV.relative_to(ROOT)}")
print("=" * 80)
print(f"1. Phân bố theo lớp khuyết tật:")
for cid in sorted(names_dict.keys()):
    cname = names_dict[cid]
    count = len(df_val_gt[df_val_gt['class_id'] == cid])
    print(f"   - [{cid}] {cname:<16}: {count:>4} boxes ({count/len(df_val_gt):>6.2%})")

print(f"\n2. Phân bố theo size bin (COCO scale imgsz=640):")
for sbin in ["small", "medium", "large"]:
    count = len(df_val_gt[df_val_gt['size_bin'] == sbin])
    print(f"   - {sbin:<8}: {count:>4} boxes ({count/len(df_val_gt):>6.2%})")
print("=" * 80)


XÁC NHẬN TẢI THÀNH CÔNG GROUND TRUTH VALIDATION: 1,860 BOXES
Đã lưu cache tại: results/val_ground_truth.csv
1. Phân bố theo lớp khuyết tật:
   - [0] missing_hole    :  300 boxes (16.13%)
   - [1] mouse_bite      :  304 boxes (16.34%)
   - [2] open_circuit    :  324 boxes (17.42%)
   - [3] short           :  304 boxes (16.34%)
   - [4] spur            :  300 boxes (16.13%)
   - [5] spurious_copper :  328 boxes (17.63%)

2. Phân bố theo size bin (COCO scale imgsz=640):
   - small   : 1570 boxes (84.41%)
   - medium  :  290 boxes (15.59%)
   - large   :    0 boxes ( 0.00%)


### 8. Xây Dựng Hàm Tính IoU và Thuật Toán Matching (Phase 4)
- **Quy tắc Matching chuẩn PASCAL VOC / COCO**:
  1. Chỉ so khớp giữa prediction và ground truth **CÙNG LỚP (`class_id`)** trên **CÙNG MỘT ẢNH**.
  2. Sắp xếp danh sách predictions theo điểm tin cậy `confidence` giảm dần (ưu tiên gán box có độ tự tin cao nhất).
  3. Với mỗi prediction, tìm ground truth chưa ghép cặp có chỉ số $IoU \ge 0.5$ cao nhất:
     - Nếu tìm thấy: Ghi nhận **True Positive (TP)**; đánh dấu ground truth này đã matched (mỗi GT chỉ match tối đa 1 lần).
     - Nếu không tìm thấy bất kỳ GT nào thỏa $IoU \ge 0.5$: Ghi nhận **False Positive (FP)**.
  4. Mọi ground truth không được bất kỳ prediction nào match sau khi duyệt hết: Ghi nhận **False Negative (FN)**.
- **Hệ thống chỉ số trích xuất**:
  - Toàn cục: $TP, FP, FN, Precision, Recall, F_1, F_2$ ($eta = 2$)
  - Theo từng lớp khuyết tật: $TP_c, FP_c, FN_c, Precision_c, Recall_c, F1_c, F2_c$
  - Theo lớp lỗi chí mạng: $Recall_{critical}$ và $Critical\ FN\ Rate = rac{FN_{open} + FN_{short}}{GT_{open} + GT_{short}}$
  - Theo nhóm kích thước: $Recall_{small}, Recall_{medium}, Recall_{large}$


In [7]:
def compute_iou(box1, box2) -> float:
    # Tính Intersection over Union (IoU) giữa hai bounding box: [xmin, ymin, xmax, ymax]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter_area = inter_w * inter_h
    
    area1 = max(0.0, (box1[2] - box1[0]) * (box1[3] - box1[1]))
    area2 = max(0.0, (box2[2] - box2[0]) * (box2[3] - box2[1]))
    union_area = area1 + area2 - inter_area
    
    return inter_area / union_area if union_area > 0.0 else 0.0


# Tiền tổ chức Ground Truth theo image_name để tăng tốc độ đối sánh O(1)
gt_by_image = defaultdict(list)
for _, r in df_val_gt.iterrows():
    gt_by_image[r["image_name"]].append({
        "class_id": int(r["class_id"]),
        "class_name": r["class_name"],
        "box": [float(r["xmin"]), float(r["ymin"]), float(r["xmax"]), float(r["ymax"])],
        "size_bin": r["size_bin"],
    })


def evaluate_threshold_matching(
    df_preds_all: pd.DataFrame,
    conf_thresh: float,
    iou_thresh: float = 0.5,
    beta: float = 2.0,
    critical_class_names: list = ["open_circuit", "short"],
) -> dict:
    # Lọc predictions theo ngưỡng conf_thresh và thực hiện đối sánh với df_val_gt.
    # Trả về bộ chỉ số hoàn chỉnh phục vụ threshold sweep.
    # 1. Lọc dự đoán theo ngưỡng tin cậy
    df_p = df_preds_all[df_preds_all["confidence"] >= conf_thresh]
    
    # Nhóm predictions theo ảnh và sắp xếp theo confidence giảm dần
    preds_by_image = defaultdict(list)
    if len(df_p) > 0:
        for _, row in df_p.sort_values(by="confidence", ascending=False).iterrows():
            preds_by_image[row["image_name"]].append({
                "class_id": int(row["class_id"]),
                "confidence": float(row["confidence"]),
                "box": [float(row["xmin"]), float(row["ymin"]), float(row["xmax"]), float(row["ymax"])],
            })
            
    # Bộ đếm tổng quát và theo lớp
    tp_global, fp_global, fn_global = 0, 0, 0
    tp_by_class = defaultdict(int)
    fp_by_class = defaultdict(int)
    fn_by_class = defaultdict(int)
    
    # Bộ đếm theo size bin
    tp_by_size = defaultdict(int)
    gt_by_size = defaultdict(int)
    
    for img_name, img_gts in gt_by_image.items():
        img_preds = preds_by_image.get(img_name, [])
        gt_matched = [False] * len(img_gts)
        
        # Thống kê tổng số GT theo size bin
        for gt in img_gts:
            gt_by_size[gt["size_bin"]] += 1
            
        # Duyệt qua từng prediction theo thứ tự confidence cao -> thấp
        for p in img_preds:
            p_cid = p["class_id"]
            p_box = p["box"]
            
            best_iou = 0.0
            best_gt_idx = -1
            
            for idx, gt in enumerate(img_gts):
                if not gt_matched[idx] and gt["class_id"] == p_cid:
                    iou = compute_iou(p_box, gt["box"])
                    if iou >= iou_thresh and iou > best_iou:
                        best_iou = iou
                        best_gt_idx = idx
            
            if best_gt_idx != -1:
                # Match thành công: TP
                tp_global += 1
                tp_by_class[p_cid] += 1
                tp_by_size[img_gts[best_gt_idx]["size_bin"]] += 1
                gt_matched[best_gt_idx] = True
            else:
                # Không tìm được GT phù hợp: FP
                fp_global += 1
                fp_by_class[p_cid] += 1
                
        # Các GT còn lại không được match: FN
        for idx, matched in enumerate(gt_matched):
            if not matched:
                fn_global += 1
                fn_by_class[img_gts[idx]["class_id"]] += 1

    # 2. Tính các chỉ số toàn cục
    precision = tp_global / (tp_global + fp_global) if (tp_global + fp_global) > 0 else 0.0
    recall = tp_global / (tp_global + fn_global) if (tp_global + fn_global) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    beta_sq = beta ** 2
    f2 = (1 + beta_sq) * precision * recall / (beta_sq * precision + recall) if (beta_sq * precision + recall) > 0 else 0.0
    
    # 3. Tính metrics cho Critical Defects
    critical_cids = [cid for cid, cname in names_dict.items() if cname in critical_class_names]
    tp_crit = sum(tp_by_class[cid] for cid in critical_cids)
    fn_crit = sum(fn_by_class[cid] for cid in critical_cids)
    fp_crit = sum(fp_by_class[cid] for cid in critical_cids)
    gt_crit = tp_crit + fn_crit
    
    recall_critical = tp_crit / gt_crit if gt_crit > 0 else 0.0
    critical_fn_rate = fn_crit / gt_crit if gt_crit > 0 else 0.0
    
    prec_crit = tp_crit / (tp_crit + fp_crit) if (tp_crit + fp_crit) > 0 else 0.0
    f2_critical = (1 + beta_sq) * prec_crit * recall_critical / (beta_sq * prec_crit + recall_critical) if (beta_sq * prec_crit + recall_critical) > 0 else 0.0

    # 4. Tính Recall theo size bin
    recall_small = tp_by_size["small"] / gt_by_size["small"] if gt_by_size["small"] > 0 else 0.0
    recall_medium = tp_by_size["medium"] / gt_by_size["medium"] if gt_by_size["medium"] > 0 else 0.0
    recall_large = tp_by_size["large"] / gt_by_size["large"] if gt_by_size["large"] > 0 else 0.0

    return {
        "confidence_threshold": conf_thresh,
        "TP": tp_global,
        "FP": fp_global,
        "FN": fn_global,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
        "critical_TP": tp_crit,
        "critical_FP": fp_crit,
        "critical_FN": fn_crit,
        "critical_recall": recall_critical,
        "critical_fn_rate": critical_fn_rate,
        "critical_f2": f2_critical,
        "recall_small": recall_small,
        "recall_medium": recall_medium,
        "recall_large": recall_large,
        "tp_by_class": dict(tp_by_class),
        "fp_by_class": dict(fp_by_class),
        "fn_by_class": dict(fn_by_class),
    }


# ── Kiểm tra thử nghiệm hàm matching trên B0 và B1 ở mốc tiêu chuẩn conf = 0.25 ──
sample_b0 = evaluate_threshold_matching(val_predictions_dict["B0"], conf_thresh=0.25)
sample_b1 = evaluate_threshold_matching(val_predictions_dict["B1"], conf_thresh=0.25)

check_df = pd.DataFrame([
    {
        "Model": "B0 (YOLOv8n)",
        "Conf": 0.25,
        "TP": sample_b0["TP"],
        "FP": sample_b0["FP"],
        "FN": sample_b0["FN"],
        "Precision": f"{sample_b0['precision']:.4f}",
        "Recall": f"{sample_b0['recall']:.4f}",
        "F1-Score": f"{sample_b0['f1']:.4f}",
        "F2-Score": f"{sample_b0['f2']:.4f}",
        "Critical Recall": f"{sample_b0['critical_recall']:.2%}",
        "Critical FN Rate": f"{sample_b0['critical_fn_rate']:.2%}",
    },
    {
        "Model": "B1 (YOLOv8n-P2)",
        "Conf": 0.25,
        "TP": sample_b1["TP"],
        "FP": sample_b1["FP"],
        "FN": sample_b1["FN"],
        "Precision": f"{sample_b1['precision']:.4f}",
        "Recall": f"{sample_b1['recall']:.4f}",
        "F1-Score": f"{sample_b1['f1']:.4f}",
        "F2-Score": f"{sample_b1['f2']:.4f}",
        "Critical Recall": f"{sample_b1['critical_recall']:.2%}",
        "Critical FN Rate": f"{sample_b1['critical_fn_rate']:.2%}",
    }
])

print("=== KIỂM THỬ THÀNH CÔNG HÀM MATCHING TẠI CONF = 0.25 ===")
print(check_df.to_string(index=False))


=== KIỂM THỬ THÀNH CÔNG HÀM MATCHING TẠI CONF = 0.25 ===
          Model  Conf   TP  FP  FN Precision Recall F1-Score F2-Score Critical Recall Critical FN Rate
   B0 (YOLOv8n)  0.25 1375 714 485    0.6582 0.7392   0.6964   0.7215          75.48%           24.52%
B1 (YOLOv8n-P2)  0.25 1177 796 683    0.5966 0.6328   0.6141   0.6252          61.31%           38.69%


### 9. Quét Lưới Confidence Threshold (0.01 đến 0.95) & Tính Toán Bộ Chỉ Số Đánh Giá (Phase 5)
- **Phương pháp thực hiện**:
  1. Với mỗi mô hình ($B_0$ và $B_1$), duyệt qua 95 mốc giá trị ngưỡng tin cậy từ `0.01` đến `0.95` (bước `0.01`).
  2. Tại mỗi ngưỡng, lọc tập dự đoán cache có điểm tin cậy $confidence \ge threshold$.
  3. Tính toán toàn bộ các chỉ số đo lường hiệu năng:
     - $Precision, Recall, F_1	ext{-Score}, F_2	ext{-Score}$ ($eta = 2$)
     - $Recall_{critical}$ và $Critical\ FN\ Rate$ (dành riêng cho 2 lớp chí mạng `open_circuit` và `short`)
     - $Recall_{small}, Recall_{medium}, Recall_{large}$ (theo chuẩn COCO size bin)
     - $Board\ False\ Accept\ Rate$ và $Board\ False\ Reject\ Rate$ (theo cấp độ bo mạch)
- **Quy tắc đánh giá cấp độ bo mạch (Board-Level)**:
  - *Ground Truth DEFECTIVE*: Ảnh chứa ít nhất 1 nhãn lỗi thực tế.
  - *Prediction DEFECTIVE*: Mô hình phát hiện ít nhất 1 box có điểm tin cậy $\ge threshold$.
  - *False Accept*: Bo mạch thực tế bị lỗi nhưng mô hình dự đoán PASS (0 phát hiện) $ightarrow$ Lọt lỗi ra dây chuyền.
  - *False Reject*: Bo mạch không có lỗi nhưng mô hình báo DEFECTIVE $ightarrow$ Báo động giả.
  - *Ràng buộc tập dữ liệu (Limitation)*: Tập validation của PKU-Market-PCB có 100% bo mạch đều chứa khuyết tật (0 bo mạch lành lặn). Do đó, mẫu số tính False Reject Rate bằng 0, giá trị này được đặt là `null` (None/NaN) theo đúng quy định khoa học.
- **Tệp xuất ra**: `results/threshold_sweep_val.csv` chứa 190 dòng dữ liệu (95 ngưỡng $	imes$ 2 mô hình).


In [8]:
val_images_set = set(df_val_gt["image_name"])
total_val_boards = len(val_images_set)

sweep_records = []

print("=" * 80)
print(f"BẮT ĐẦU QUÉT LƯỚI THRESHOLD TRÊN VALIDATION ({len(conf_values)} MỐC TỪ 0.01 ĐẾN 0.95)...")
print("=" * 80)

for model_code in ["B0", "B1"]:
    df_model_preds = val_predictions_dict[model_code]
    print(f"\n🚀 Đang quét mô hình {model_code} qua {len(conf_values)} ngưỡng...")
    
    for conf in conf_values:
        # 1. Lọc dự đoán theo ngưỡng
        df_p = df_model_preds[df_model_preds["confidence"] >= conf]
        
        # 2. Nhóm predictions theo ảnh và lấy danh sách ảnh phát hiện có lỗi
        preds_by_img = defaultdict(list)
        detected_images = set()
        if len(df_p) > 0:
            for _, row in df_p.sort_values(by="confidence", ascending=False).iterrows():
                preds_by_img[row["image_name"]].append({
                    "class_id": int(row["class_id"]),
                    "box": [float(row["xmin"]), float(row["ymin"]), float(row["xmax"]), float(row["ymax"])],
                })
                detected_images.add(row["image_name"])
                
        # 3. Matching với Ground Truth
        tp, fp, fn = 0, 0, 0
        tp_small, gt_small = 0, 1570
        tp_medium, gt_medium = 0, 290
        tp_crit, fn_crit = 0, 0
        
        for img_name, img_gts in gt_by_image.items():
            img_preds = preds_by_img.get(img_name, [])
            gt_matched = [False] * len(img_gts)
            
            for p in img_preds:
                p_cid = p["class_id"]
                p_box = p["box"]
                best_iou = 0.0
                best_idx = -1
                
                for idx, gt in enumerate(img_gts):
                    if not gt_matched[idx] and gt["class_id"] == p_cid:
                        iou = compute_iou(p_box, gt["box"])
                        if iou >= iou_threshold and iou > best_iou:
                            best_iou = iou
                            best_idx = idx
                            
                if best_idx != -1:
                    tp += 1
                    gt_matched[best_idx] = True
                    s = img_gts[best_idx]["size_bin"]
                    if s == "small":
                        tp_small += 1
                    elif s == "medium":
                        tp_medium += 1
                    if img_gts[best_idx]["class_name"] in critical_classes:
                        tp_crit += 1
                else:
                    fp += 1
                    
            for idx, m in enumerate(gt_matched):
                if not m:
                    fn += 1
                    if img_gts[idx]["class_name"] in critical_classes:
                        fn_crit += 1
                        
        # 4. Tính toán các metric
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        
        beta_sq = beta ** 2
        f2 = (1 + beta_sq) * prec * rec / (beta_sq * prec + rec) if (beta_sq * prec + rec) > 0 else 0.0
        
        gt_crit_total = tp_crit + fn_crit
        rec_crit = tp_crit / gt_crit_total if gt_crit_total > 0 else 0.0
        fn_rate_crit = fn_crit / gt_crit_total if gt_crit_total > 0 else 0.0
        
        rec_small = tp_small / gt_small if gt_small > 0 else 0.0
        rec_medium = tp_medium / gt_medium if gt_medium > 0 else 0.0
        rec_large = 0.0  # Tập validation không có box large (>96^2)
        
        # 5. Tính chỉ số cấp độ bo mạch (Board-level)
        # False Accept: Bo mạch thực tế bị lỗi nhưng model dự đoán không có lỗi (PASS)
        false_accept_count = len(val_images_set - detected_images)
        board_far = false_accept_count / total_val_boards
        board_frr = None  # Giới hạn dataset: không có bo mạch defect-free
        
        sweep_records.append({
            "model_code": model_code,
            "conf": round(conf, 2),
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "f1": round(f1, 4),
            "f2": round(f2, 4),
            "recall_critical": round(rec_crit, 4),
            "critical_fn_rate": round(fn_rate_crit, 4),
            "recall_small": round(rec_small, 4),
            "recall_medium": round(rec_medium, 4),
            "recall_large": round(rec_large, 4),
            "board_false_accept_rate": round(board_far, 4),
            "board_false_reject_rate": board_frr,
        })

df_sweep = pd.DataFrame(sweep_records)

# Lưu file CSV kết quả quét
SWEEP_CSV_PATH = ROOT / "results" / "threshold_sweep_val.csv"
df_sweep.to_csv(SWEEP_CSV_PATH, index=False)
print(f"\n✅ ĐÃ HOÀN TẤT VÀ LƯU KẾT QUẢ VÀO: {SWEEP_CSV_PATH.relative_to(ROOT)}")
print(f"   Tổng số dòng dữ liệu: {len(df_sweep)} dòng (2 mô hình x {len(conf_values)} ngưỡng)")

# ── Tổng hợp các điểm tối ưu theo tiêu chí ──────────────────────────────────
print("\n" + "=" * 80)
print("BẢNG TỔNG HỢP CÁC ĐIỂM NGƯỠNG TỐI ƯU TRÊN VALIDATION (F1-MAX VS F2-MAX)")
print("=" * 80)

opt_summary = []
for m_code in ["B0", "B1"]:
    sub = df_sweep[df_sweep["model_code"] == m_code]
    
    # 1. Điểm cân bằng F1 tối đa
    best_f1_row = sub.loc[sub["f1"].idxmax()]
    # 2. Điểm ưu tiên Recall F2 tối đa
    best_f2_row = sub.loc[sub["f2"].idxmax()]
    # 3. Điểm ngưỡng tiêu chuẩn 0.25
    def_row = sub[sub["conf"] == 0.25].iloc[0]
    
    opt_summary.append({
        "Model": m_code,
        "Criterion": "Default (conf=0.25)",
        "Optimal Conf": def_row["conf"],
        "Precision": def_row["precision"],
        "Recall": def_row["recall"],
        "F1-Score": def_row["f1"],
        "F2-Score": def_row["f2"],
        "Critical Recall": f"{def_row['recall_critical']:.2%}",
        "Critical FN Rate": f"{def_row['critical_fn_rate']:.2%}",
        "Board Escape (FAR)": f"{def_row['board_false_accept_rate']:.2%}",
    })
    opt_summary.append({
        "Model": m_code,
        "Criterion": "F1-Max (Balanced)",
        "Optimal Conf": best_f1_row["conf"],
        "Precision": best_f1_row["precision"],
        "Recall": best_f1_row["recall"],
        "F1-Score": best_f1_row["f1"],
        "F2-Score": best_f1_row["f2"],
        "Critical Recall": f"{best_f1_row['recall_critical']:.2%}",
        "Critical FN Rate": f"{best_f1_row['critical_fn_rate']:.2%}",
        "Board Escape (FAR)": f"{best_f1_row['board_false_accept_rate']:.2%}",
    })
    opt_summary.append({
        "Model": m_code,
        "Criterion": "F2-Max (Recall-Prioritized)",
        "Optimal Conf": best_f2_row["conf"],
        "Precision": best_f2_row["precision"],
        "Recall": best_f2_row["recall"],
        "F1-Score": best_f2_row["f1"],
        "F2-Score": best_f2_row["f2"],
        "Critical Recall": f"{best_f2_row['recall_critical']:.2%}",
        "Critical FN Rate": f"{best_f2_row['critical_fn_rate']:.2%}",
        "Board Escape (FAR)": f"{best_f2_row['board_false_accept_rate']:.2%}",
    })

df_opt = pd.DataFrame(opt_summary)
print(df_opt.to_string(index=False))
print("=" * 80)


BẮT ĐẦU QUÉT LƯỚI THRESHOLD TRÊN VALIDATION (95 MỐC TỪ 0.01 ĐẾN 0.95)...

🚀 Đang quét mô hình B0 qua 95 ngưỡng...



🚀 Đang quét mô hình B1 qua 95 ngưỡng...



✅ ĐÃ HOÀN TẤT VÀ LƯU KẾT QUẢ VÀO: results/threshold_sweep_val.csv
   Tổng số dòng dữ liệu: 190 dòng (2 mô hình x 95 ngưỡng)

BẢNG TỔNG HỢP CÁC ĐIỂM NGƯỠNG TỐI ƯU TRÊN VALIDATION (F1-MAX VS F2-MAX)
Model                   Criterion  Optimal Conf  Precision  Recall  F1-Score  F2-Score Critical Recall Critical FN Rate Board Escape (FAR)
   B0         Default (conf=0.25)          0.25     0.6582  0.7392    0.6964    0.7215          75.48%           24.52%              0.00%
   B0           F1-Max (Balanced)          0.42     0.7860  0.6634    0.7195    0.6848          68.15%           31.85%              0.27%
   B0 F2-Max (Recall-Prioritized)          0.20     0.6356  0.7559    0.6906    0.7283          76.91%           23.09%              0.00%
   B1         Default (conf=0.25)          0.25     0.5966  0.6328    0.6141    0.6252          61.31%           38.69%              0.00%
   B1           F1-Max (Balanced)          0.24     0.5903  0.6414    0.6148    0.6305          62.26%     

### 10. Trực Quan Hóa Đồ Thị Đường Cong Threshold Sweep
Biểu diễn trực quan biến thiên của $F_1, F_2, Precision, Recall$, tỷ lệ sót lỗi chí mạng ($Critical\ FN\ Rate$) và tỷ lệ lọt bo mạch lỗi ($Board\ False\ Accept\ Rate$) theo các mốc confidence threshold từ 0.01 đến 0.95.


In [9]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=150)
plt.subplots_adjust(hspace=0.28, wspace=0.2)

df_b0 = df_sweep[df_sweep["model_code"] == "B0"]
df_b1 = df_sweep[df_sweep["model_code"] == "B1"]

# Subplot 1: F1-Score & F2-Score vs Confidence
ax = axes[0, 0]
ax.plot(df_b0["conf"], df_b0["f1"], color="#1f77b4", linewidth=2, label="B0: F1-Score")
ax.plot(df_b0["conf"], df_b0["f2"], color="#1f77b4", linestyle="--", linewidth=1.8, label="B0: F2-Score (beta=2)")
ax.plot(df_b1["conf"], df_b1["f1"], color="#d62728", linewidth=2, label="B1_P2: F1-Score")
ax.plot(df_b1["conf"], df_b1["f2"], color="#d62728", linestyle="--", linewidth=1.8, label="B1_P2: F2-Score (beta=2)")

# Đánh dấu các đỉnh F1-Max và F2-Max
b0_f1_max = df_b0.loc[df_b0["f1"].idxmax()]
b1_f1_max = df_b1.loc[df_b1["f1"].idxmax()]
ax.scatter([b0_f1_max["conf"]], [b0_f1_max["f1"]], color="#1f77b4", s=80, zorder=5)
ax.scatter([b1_f1_max["conf"]], [b1_f1_max["f1"]], color="#d62728", s=80, zorder=5)

ax.set_title("Biến Thiên F1-Score & F2-Score Theo Ngưỡng Tin Cậy", fontsize=12, fontweight="bold")
ax.set_xlabel("Confidence Threshold", fontsize=10)
ax.set_ylabel("Score", fontsize=10)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=9, loc="lower left")

# Subplot 2: Precision & Recall Curves
ax = axes[0, 1]
ax.plot(df_b0["conf"], df_b0["precision"], color="#1f77b4", linestyle=":", linewidth=2, label="B0: Precision")
ax.plot(df_b0["conf"], df_b0["recall"], color="#1f77b4", linewidth=2, label="B0: Recall")
ax.plot(df_b1["conf"], df_b1["precision"], color="#d62728", linestyle=":", linewidth=2, label="B1_P2: Precision")
ax.plot(df_b1["conf"], df_b1["recall"], color="#d62728", linewidth=2, label="B1_P2: Recall")

ax.set_title("Đường Cong Precision & Recall Theo Ngưỡng Tin Cậy", fontsize=12, fontweight="bold")
ax.set_xlabel("Confidence Threshold", fontsize=10)
ax.set_ylabel("Metric Value", fontsize=10)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=9, loc="center left")

# Subplot 3: Critical Defect False Negative Rate (open_circuit + short)
ax = axes[1, 0]
ax.plot(df_b0["conf"], df_b0["critical_fn_rate"], color="#1f77b4", linewidth=2.2, label="B0: Critical FN Rate")
ax.plot(df_b1["conf"], df_b1["critical_fn_rate"], color="#d62728", linewidth=2.2, label="B1_P2: Critical FN Rate")
ax.axhline(0.20, color="gray", linestyle="--", alpha=0.7, label="Mục tiêu tối thiểu FN Rate <= 20%")

ax.set_title("Tỷ Lệ Bỏ Sót Lỗi Chí Mạng (Critical FN Rate)", fontsize=12, fontweight="bold")
ax.set_xlabel("Confidence Threshold", fontsize=10)
ax.set_ylabel("False Negative Rate (càng thấp càng an toàn)", fontsize=10)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=9, loc="upper left")

# Subplot 4: Board-Level False Accept Rate (Tỷ Lệ Lọt Bo Mạch Lỗi)
ax = axes[1, 1]
ax.plot(df_b0["conf"], df_b0["board_false_accept_rate"], color="#1f77b4", linewidth=2.2, label="B0: Board False Accept Rate")
ax.plot(df_b1["conf"], df_b1["board_false_accept_rate"], color="#d62728", linewidth=2.2, label="B1_P2: Board False Accept Rate")

ax.set_title("Tỷ Lệ Lọt Bo Mạch Lỗi Cấp Hệ Thống (Board False Accept Rate)", fontsize=12, fontweight="bold")
ax.set_xlabel("Confidence Threshold", fontsize=10)
ax.set_ylabel("Board False Accept Rate (Escape Rate)", fontsize=10)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 0.15)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=9, loc="upper left")

# Lưu biểu đồ phân tích
SWEEP_CURVES_PNG = ROOT / "results" / "threshold_sweep_curves.png"
plt.savefig(SWEEP_CURVES_PNG, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"✅ Đã xuất biểu đồ trực quan hóa đường cong sweep sang: {SWEEP_CURVES_PNG.relative_to(ROOT)}")


✅ Đã xuất biểu đồ trực quan hóa đường cong sweep sang: results/threshold_sweep_curves.png


### 11. Xuất 3 Biểu Đồ Chuẩn Hóa Phục Vụ Báo Cáo Luận Văn (Chương 4) - Phase 6
Theo yêu cầu phân tích chuyên sâu cho Chương 4 (Thực nghiệm & Đánh giá kết quả), hệ thống xuất riêng 3 biểu đồ độ phân giải cao:
1. **Biểu đồ 1 (`results/threshold_precision_recall_vs_conf.png`)**: Biến thiên Precision & Recall theo confidence threshold cho cả hai mô hình $B_0$ và $B_1$ (tách subplot so sánh trực tiếp).
2. **Biểu đồ 2 (`results/threshold_f2_vs_conf.png`)**: Biến thiên $F_2$-Score theo confidence threshold, đánh dấu điểm tối ưu $conf^*$ của $B_0$ ($conf=0.20$) và $B_1$ ($conf=0.13$).
3. **Biểu đồ 3 (`results/threshold_board_fa_fr_vs_conf.png`)**: Tỷ lệ lọt bo mạch lỗi cấp hệ thống ($Board\ False\ Accept\ Rate$), kèm ghi chú minh bạch về giới hạn tập dữ liệu không có bo lành ($False\ Reject\ Rate = null$).


In [10]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150, sharey=True)

df_b0 = df_sweep[df_sweep["model_code"] == "B0"]
df_b1 = df_sweep[df_sweep["model_code"] == "B1"]

# Subplot 1: B0
ax1.plot(df_b0["conf"], df_b0["precision"], color="#1f77b4", linestyle="--", linewidth=2.2, label="Precision")
ax1.plot(df_b0["conf"], df_b0["recall"], color="#2ca02c", linestyle="-", linewidth=2.2, label="Recall")
b0_f1_opt = df_b0.loc[df_b0["f1"].idxmax()]
ax1.axvline(b0_f1_opt["conf"], color="#ff7f0e", linestyle=":", linewidth=1.5, label=f"F1-Max (conf={b0_f1_opt['conf']:.2f})")
ax1.axvline(0.25, color="gray", linestyle="-.", linewidth=1.2, alpha=0.7, label="Default (conf=0.25)")
ax1.set_title("Mô hình B0 (YOLOv8n)", fontsize=13, fontweight="bold", pad=10)
ax1.set_xlabel("Confidence Threshold", fontsize=11)
ax1.set_ylabel("Score", fontsize=11)
ax1.set_xlim(0.0, 1.0)
ax1.set_ylim(0.0, 1.05)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(fontsize=9.5, loc="center left")

# Subplot 2: B1
ax2.plot(df_b1["conf"], df_b1["precision"], color="#d62728", linestyle="--", linewidth=2.2, label="Precision")
ax2.plot(df_b1["conf"], df_b1["recall"], color="#2ca02c", linestyle="-", linewidth=2.2, label="Recall")
b1_f1_opt = df_b1.loc[df_b1["f1"].idxmax()]
ax2.axvline(b1_f1_opt["conf"], color="#ff7f0e", linestyle=":", linewidth=1.5, label=f"F1-Max (conf={b1_f1_opt['conf']:.2f})")
ax2.axvline(0.25, color="gray", linestyle="-.", linewidth=1.2, alpha=0.7, label="Default (conf=0.25)")
ax2.set_title("Mô hình B1 (YOLOv8n-P2)", fontsize=13, fontweight="bold", pad=10)
ax2.set_xlabel("Confidence Threshold", fontsize=11)
ax2.set_xlim(0.0, 1.0)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(fontsize=9.5, loc="center left")

fig.suptitle("Biến Thiên Precision và Recall Theo Ngưỡng Tin Cậy (B0 vs B1)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
FIG1_PATH = ROOT / "results" / "threshold_precision_recall_vs_conf.png"
plt.savefig(FIG1_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✅ Đã lưu Biểu đồ 1: {FIG1_PATH.relative_to(ROOT)}")


✅ Đã lưu Biểu đồ 1: results/threshold_precision_recall_vs_conf.png


In [11]:
fig, ax = plt.subplots(figsize=(9.5, 6), dpi=150)

ax.plot(df_b0["conf"], df_b0["f2"], color="#1f77b4", linewidth=2.5, label="B0: YOLOv8n ($F_2$-Score)")
ax.plot(df_b1["conf"], df_b1["f2"], color="#d62728", linewidth=2.5, label="B1: YOLOv8n-P2 ($F_2$-Score)")

# Đánh dấu các đỉnh F2-Max tối ưu
b0_f2_opt = df_b0.loc[df_b0["f2"].idxmax()]
b1_f2_opt = df_b1.loc[df_b1["f2"].idxmax()]

ax.scatter([b0_f2_opt["conf"]], [b0_f2_opt["f2"]], color="#1f77b4", s=110, zorder=5, edgecolor="black")
ax.scatter([b1_f2_opt["conf"]], [b1_f2_opt["f2"]], color="#d62728", s=110, zorder=5, edgecolor="black")

ax.axvline(b0_f2_opt["conf"], color="#1f77b4", linestyle="--", linewidth=1.3, alpha=0.7)
ax.axvline(b1_f2_opt["conf"], color="#d62728", linestyle="--", linewidth=1.3, alpha=0.7)

# Ghi chú trực quan hóa
ax.annotate(
    f"B0 Tối Ưu F2: {b0_f2_opt['f2']:.4f}\ntại conf = {b0_f2_opt['conf']:.2f}",
    xy=(b0_f2_opt["conf"], b0_f2_opt["f2"]),
    xytext=(b0_f2_opt["conf"] + 0.08, b0_f2_opt["f2"] + 0.04),
    arrowprops=dict(facecolor="#1f77b4", shrink=0.08, width=1.5, headwidth=6),
    bbox=dict(boxstyle="round,pad=0.4", fc="#e8f4fd", ec="#1f77b4", alpha=0.9),
    fontsize=9.5, fontweight="bold"
)

ax.annotate(
    f"B1 Tối Ưu F2: {b1_f2_opt['f2']:.4f}\ntại conf = {b1_f2_opt['conf']:.2f}",
    xy=(b1_f2_opt["conf"], b1_f2_opt["f2"]),
    xytext=(b1_f2_opt["conf"] + 0.08, b1_f2_opt["f2"] - 0.08),
    arrowprops=dict(facecolor="#d62728", shrink=0.08, width=1.5, headwidth=6),
    bbox=dict(boxstyle="round,pad=0.4", fc="#fde8e8", ec="#d62728", alpha=0.9),
    fontsize=9.5, fontweight="bold"
)

ax.axvline(0.25, color="gray", linestyle="-.", linewidth=1.2, alpha=0.6, label="Ngưỡng mặc định (conf=0.25)")

ax.set_title("Đường Cong Chỉ Số F2-Score (β = 2) Theo Ngưỡng Tin Cậy Confidence", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Confidence Threshold (Ngưỡng tin cậy)", fontsize=11)
ax.set_ylabel("F2-Score (Trọng số ưu tiên Recall)", fontsize=11)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 0.85)
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=10, loc="lower left")

plt.tight_layout()
FIG2_PATH = ROOT / "results" / "threshold_f2_vs_conf.png"
plt.savefig(FIG2_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✅ Đã lưu Biểu đồ 2: {FIG2_PATH.relative_to(ROOT)}")


✅ Đã lưu Biểu đồ 2: results/threshold_f2_vs_conf.png


In [12]:
fig, ax = plt.subplots(figsize=(9.5, 6), dpi=150)

ax.plot(df_b0["conf"], df_b0["board_false_accept_rate"], color="#1f77b4", linewidth=2.5, label="B0: Board False Accept Rate (FAR)")
ax.plot(df_b1["conf"], df_b1["board_false_accept_rate"], color="#d62728", linewidth=2.5, linestyle="--", label="B1: Board False Accept Rate (FAR)")

# Vùng an toàn tuyệt đối
ax.axvspan(0.0, 0.35, color="green", alpha=0.08, label="Vùng an toàn tuyệt đối (FAR = 0.00%)")
ax.axvline(0.25, color="gray", linestyle="-.", linewidth=1.2, alpha=0.7, label="Ngưỡng mặc định (conf=0.25)")

# Hộp văn bản ghi nhận limitation về False Reject Rate
limitation_text = (
    "LƯU Ý ĐẶC TÍNH TẬP DỮ LIỆU:\n"
    "• False Reject Rate (FRR) = null (N/A) do toàn bộ 368 ảnh trong tập validation\n"
    "  của PKU-Market-PCB đều chứa khuyết tật (0 bo mạch lành lặn).\n"
    "• Board False Accept Rate (FAR) phản ánh tỷ lệ bo mạch lỗi nhưng bị bỏ sót (PASS).\n"
    "• Với conf <= 0.35: Cả B0 và B1 đều đạt FAR = 0.00% (an toàn tuyệt đối)."
)
ax.text(
    0.04, 0.72, limitation_text,
    transform=ax.transAxes,
    fontsize=9,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.5", fc="#fff9e6", ec="#d48800", alpha=0.95),
)

ax.set_title("Tỷ Lệ Bỏ Lọt Bo Mạch Lỗi Cấp Hệ Thống (Board False Accept Rate)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Confidence Threshold", fontsize=11)
ax.set_ylabel("Tỷ lệ lọt bo mạch lỗi (FAR / Escape Rate)", fontsize=11)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(-0.005, 0.15)
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=9.5, loc="center right")

plt.tight_layout()
FIG3_PATH = ROOT / "results" / "threshold_board_fa_fr_vs_conf.png"
plt.savefig(FIG3_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✅ Đã lưu Biểu đồ 3: {FIG3_PATH.relative_to(ROOT)}")


✅ Đã lưu Biểu đồ 3: results/threshold_board_fa_fr_vs_conf.png


### 12. Lựa Chọn Và Khóa Ngưỡng Tối Ưu (Threshold Rule Locking) - Phase 7
- **Đánh giá 2 quy tắc ứng viên (Candidate Rules)**:
  1. **Recall-Target Rule**: Ngưỡng lớn nhất thỏa mãn $Recall_{critical} \ge R_{target}$ với $R_{target} = 0.95$.
     - *Thẩm định thực nghiệm*: Trên tập validation, độ nhạy cực đại đối với 2 lớp lỗi chí mạng (`open_circuit`, `short`) của $B_0$ đạt $86.62\%$ và của $B_1$ đạt $79.78\%$ (tại $conf = 0.01$).
     - *Kết luận*: Không mô hình nào đạt được $Recall_{critical} \ge 0.95$. Do đó, **quy tắc Recall-Target (0.95) là KHÔNG KHẢ THI (not feasible) trên tập validation**.
  2. **Max-$F_2$ Rule ($eta = 2$)**: Chọn ngưỡng tin cậy đạt cực đại $F_2$-Score (ưu tiên phạt nặng lỗi bỏ sót False Negative). Nếu xảy ra hòa điểm (tie), chọn ngưỡng thấp hơn để bảo toàn Recall cao hơn.
- **Quyết định khóa chính thức**:
  - `selected_rule = "max_f2"`
  - Ngưỡng tối ưu $B_0$: $	au_{B0} = 0.20$ ($F_2 = 0.7283, Recall_{critical} = 76.91\%, FAR = 0.00\%$)
  - Ngưỡng tối ưu $B_1$: $	au_{B1} = 0.13$ ($F_2 = 0.6460, Recall_{critical} = 70.06\%, FAR = 0.00\%$)
  - Hai mô hình được gán hai ngưỡng riêng biệt, không dùng chung threshold.
- **Tệp cấu hình được tạo**: `configs/threshold.yaml`.


In [13]:
R_TARGET = 0.95

# 1. Đánh giá Recall-Target Rule
candidates_dict = {}

for m_code in ["B0", "B1"]:
    sub = df_sweep[df_sweep["model_code"] == m_code].copy()
    
    # Tìm Max F2 (nếu hòa điểm, lấy conf nhỏ nhất để ưu tiên recall)
    max_f2_val = sub["f2"].max()
    ties = sub[sub["f2"] == max_f2_val].sort_values("conf", ascending=True)
    best_f2_row = ties.iloc[0]
    max_f2_tau = float(best_f2_row["conf"])
    
    # Tìm Recall-Target Rule (conf lớn nhất sao cho recall_critical >= R_TARGET)
    target_matches = sub[sub["recall_critical"] >= R_TARGET].sort_values("conf", ascending=False)
    if len(target_matches) > 0:
        recall_target_tau = float(target_matches.iloc[0]["conf"])
        recall_target_note = f"Feasible: achieved at conf={recall_target_tau:.2f}"
    else:
        recall_target_tau = None
        max_crit = sub["recall_critical"].max()
        recall_target_note = f"Not feasible on validation (max recall_critical = {max_crit:.4f} < {R_TARGET})"
        
    candidates_dict[m_code] = {
        "max_f2_tau": max_f2_tau,
        "max_f2_score": float(max_f2_val),
        "recall_target_tau": recall_target_tau,
        "recall_target_note": recall_target_note,
        "best_row": best_f2_row,
    }

# 2. Quyết định Rule chính thức
if all(c["recall_target_tau"] is not None for c in candidates_dict.values()):
    selected_rule = "recall_target"
else:
    selected_rule = "max_f2"
    print(f"⚠️  Quy tắc Recall-Target ({R_TARGET}) không khả thi trên tập validation:")
    for m, c in candidates_dict.items():
        print(f"   - {m}: {c['recall_target_note']}")
    print(f"👉 Chọn quy tắc chính thức: selected_rule = '{selected_rule}'")

tau_b0 = candidates_dict["B0"]["max_f2_tau"]
tau_b1 = candidates_dict["B1"]["max_f2_tau"]

row_b0 = candidates_dict["B0"]["best_row"]
row_b1 = candidates_dict["B1"]["best_row"]

# 3. Tạo cấu trúc configs/threshold.yaml
threshold_config = {
    "selected_rule": selected_rule,
    "chosen_on": "validation",
    "beta": 2,
    "iou_threshold": 0.5,
    "critical_classes": ["open_circuit", "short"],
    "board_decision_rule": "defective_if_any_detection_conf_ge_tau",
    "tau": {
        "B0": tau_b0,
        "B1": tau_b1,
    },
    "candidates": {
        "B0": {
            "max_f2_tau": tau_b0,
            "recall_target_tau": candidates_dict["B0"]["recall_target_tau"],
            "recall_target_note": candidates_dict["B0"]["recall_target_note"],
        },
        "B1": {
            "max_f2_tau": tau_b1,
            "recall_target_tau": candidates_dict["B1"]["recall_target_tau"],
            "recall_target_note": candidates_dict["B1"]["recall_target_note"],
        },
    },
    "metrics_at_selected_tau": {
        "B0": {
            "conf": float(row_b0["conf"]),
            "precision": float(row_b0["precision"]),
            "recall": float(row_b0["recall"]),
            "f1": float(row_b0["f1"]),
            "f2": float(row_b0["f2"]),
            "recall_critical": float(row_b0["recall_critical"]),
            "critical_fn_rate": float(row_b0["critical_fn_rate"]),
            "board_false_accept_rate": float(row_b0["board_false_accept_rate"]),
        },
        "B1": {
            "conf": float(row_b1["conf"]),
            "precision": float(row_b1["precision"]),
            "recall": float(row_b1["recall"]),
            "f1": float(row_b1["f1"]),
            "f2": float(row_b1["f2"]),
            "recall_critical": float(row_b1["recall_critical"]),
            "critical_fn_rate": float(row_b1["critical_fn_rate"]),
            "board_false_accept_rate": float(row_b1["board_false_accept_rate"]),
        },
    },
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "note": "test set was not used for threshold selection",
}

# 4. Lưu ra file configs/threshold.yaml
CONFIG_THRESHOLD_PATH = ROOT / "configs" / "threshold.yaml"
with open(CONFIG_THRESHOLD_PATH, "w", encoding="utf-8") as fp:
    yaml.dump(threshold_config, fp, default_flow_style=False, sort_keys=False, allow_unicode=True)

print("\n" + "=" * 80)
print(f"✅ ĐÃ TẠO VÀ KHÓA THÀNH CÔNG: {CONFIG_THRESHOLD_PATH.relative_to(ROOT)}")
print("=" * 80)
with open(CONFIG_THRESHOLD_PATH, "r", encoding="utf-8") as fp:
    print(fp.read())
print("=" * 80)

# 5. Ghi nhận nhật ký quyết định kỹ thuật docs/decision_log.md
decision_log_path = ROOT / "docs" / "decision_log.md"
now_date = datetime.now().strftime("%Y-%m-%d")

log_entry = f'''
## {now_date}: Khóa Ngưỡng Tối Ưu Trên Validation (Giai Đoạn 5 - Tuần 9)

- Quyết định: Khóa chính thức bộ ngưỡng tin cậy tối ưu theo quy tắc Max-F2 (beta=2) trên tập validation.
- Quy tắc được chọn (`selected_rule`): `max_f2`.
  - Lý do: Quy tắc Recall-Target (R_target = 0.95) không khả thi do độ nhạy cực đại của B0 chỉ đạt 86.62% và B1 đạt 79.78% (ngay cả tại conf=0.01).
  - Ngưỡng tối ưu B0: `tau_B0 = {tau_b0}` (Precision={row_b0['precision']:.4f}, Recall={row_b0['recall']:.4f}, F2={row_b0['f2']:.4f}, Critical Recall={row_b0['recall_critical']:.2%}, FAR={row_b0['board_false_accept_rate']:.2%}).
  - Ngưỡng tối ưu B1: `tau_B1 = {tau_b1}` (Precision={row_b1['precision']:.4f}, Recall={row_b1['recall']:.4f}, F2={row_b1['f2']:.4f}, Critical Recall={row_b1['recall_critical']:.2%}, FAR={row_b1['board_false_accept_rate']:.2%}).
- Tính độc lập của ngưỡng: Hai mô hình sử dụng hai ngưỡng riêng biệt phản ánh đặc tính kiến trúc (B0 dùng {tau_b0}, B1-P2 dùng {tau_b1}).
- Cấu hình lưu trữ: `configs/threshold.yaml`.
- Ràng buộc kiểm chứng: Toàn bộ quá trình chọn ngưỡng chỉ thực hiện trên tập `val.txt`; tuyệt đối **không sử dụng tập test (`data/splits/test.txt`)**. Tập test chỉ được đánh giá ở Tuần 10.
'''

with open(decision_log_path, "a", encoding="utf-8") as fp:
    fp.write(log_entry)

print(f"Đã cập nhật nhật ký quyết định tại: {decision_log_path.relative_to(ROOT)}")


⚠️  Quy tắc Recall-Target (0.95) không khả thi trên tập validation:
   - B0: Not feasible on validation (max recall_critical = 0.8662 < 0.95)
   - B1: Not feasible on validation (max recall_critical = 0.7978 < 0.95)
👉 Chọn quy tắc chính thức: selected_rule = 'max_f2'



✅ ĐÃ TẠO VÀ KHÓA THÀNH CÔNG: configs/threshold.yaml
selected_rule: max_f2
chosen_on: validation
beta: 2
iou_threshold: 0.5
critical_classes:
- open_circuit
- short
board_decision_rule: defective_if_any_detection_conf_ge_tau
tau:
  B0: 0.2
  B1: 0.13
candidates:
  B0:
    max_f2_tau: 0.2
    recall_target_tau: null
    recall_target_note: Not feasible on validation (max recall_critical = 0.8662 <
      0.95)
  B1:
    max_f2_tau: 0.13
    recall_target_tau: null
    recall_target_note: Not feasible on validation (max recall_critical = 0.7978 <
      0.95)
metrics_at_selected_tau:
  B0:
    conf: 0.2
    precision: 0.6356
    recall: 0.7559
    f1: 0.6906
    f2: 0.7283
    recall_critical: 0.7691
    critical_fn_rate: 0.2309
    board_false_accept_rate: 0.0
  B1:
    conf: 0.13
    precision: 0.4755
    recall: 0.7097
    f1: 0.5695
    f2: 0.646
    recall_critical: 0.7006
    critical_fn_rate: 0.2994
    board_false_accept_rate: 0.0
locked_at: '2026-09-25T06:05:07.632591+00:00'
note: